In [ ]:
# Example 1: Public API surface — top-level client + model classes are importable.
from azure.ai.textanalytics import (
    TextAnalyticsClient,
    TextAnalyticsApiVersion,
    DetectedLanguage,
    CategorizedEntity,
    LinkedEntity,
    SentimentConfidenceScores,
    PiiEntityCategory,
    PiiEntityDomain,
)

assert TextAnalyticsClient is not None
assert TextAnalyticsApiVersion is not None
print("TextAnalyticsClient:", TextAnalyticsClient.__name__)
print("Available API versions:", [v.value for v in TextAnalyticsApiVersion])
print("PiiEntityDomain values:", list(PiiEntityDomain))


In [ ]:
# Example 2: Construct a client with a fake key — exercises constructor + credential plumbing.
# No network call is made by __init__; this validates the SDK's auth wiring.
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

endpoint = "https://example.cognitiveservices.azure.com/"
credential = AzureKeyCredential("fake-key-for-smoke-test")
client = TextAnalyticsClient(endpoint=endpoint, credential=credential)

assert client is not None
# The client should expose the documented methods.
for method in ("detect_language", "recognize_entities", "recognize_linked_entities",
               "recognize_pii_entities", "analyze_sentiment", "extract_key_phrases"):
    assert hasattr(client, method), f"missing method: {method}"
    print(f"  client.{method}")
client.close()


In [ ]:
# Example 3: Document input models — DetectLanguageInput / TextDocumentInput.
from azure.ai.textanalytics import DetectLanguageInput, TextDocumentInput

lang_doc = DetectLanguageInput(id="1", text="Bonjour tout le monde", country_hint="FR")
assert lang_doc.id == "1"
assert lang_doc.text == "Bonjour tout le monde"
assert lang_doc.country_hint == "FR"
print("DetectLanguageInput:", lang_doc)

text_doc = TextDocumentInput(id="2", text="The quick brown fox", language="en")
assert text_doc.id == "2"
assert text_doc.language == "en"
print("TextDocumentInput:", text_doc)


In [ ]:
# Example 4: Action models — exercise the analyze_actions parameter classes.
from azure.ai.textanalytics import (
    RecognizeEntitiesAction,
    RecognizePiiEntitiesAction,
    ExtractKeyPhrasesAction,
    AnalyzeSentimentAction,
    RecognizeLinkedEntitiesAction,
)

actions = [
    RecognizeEntitiesAction(model_version="latest"),
    RecognizePiiEntitiesAction(model_version="latest"),
    ExtractKeyPhrasesAction(),
    AnalyzeSentimentAction(show_opinion_mining=True),
    RecognizeLinkedEntitiesAction(),
]
for a in actions:
    print(type(a).__name__, "->", a)
assert actions[0].model_version == "latest"
assert actions[3].show_opinion_mining is True
assert actions[2].model_version is None  # options left unset stay None

In [ ]:
# Example 5: Async client surface — the aio client exposes the same operations as coroutines.
import inspect

from azure.ai.textanalytics.aio import TextAnalyticsClient as AsyncTextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

async_client = AsyncTextAnalyticsClient(
    endpoint="https://example.cognitiveservices.azure.com/",
    credential=AzureKeyCredential("fake-key"),
)
for method in ("detect_language", "recognize_entities", "analyze_sentiment",
               "extract_key_phrases", "begin_analyze_actions"):
    assert inspect.iscoroutinefunction(getattr(async_client, method)), f"async_client.{method} is not async"
    print(f"  async_client.{method}")
await async_client.close()